# federated learning with flower

same problem as the baseline notebook, but each device trains locally now and only
sends model weights to a central server, not raw data. this is actually the point
of the project - checking what you give up (if anything) by going federated instead
of centralized

In [1]:
!pip install -q flwr

import flwr as fl
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from collections import OrderedDict

device_torch = "cpu"

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/nbaiot-project/processed/combined_labeled.csv")
feature_cols = [c for c in df.columns if c not in ["label", "attack_type", "device"]]
DEVICES = df["device"].unique().tolist()
DEVICES

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 M

['Danmini_Doorbell',
 'Ecobee_Thermostat',
 'Philips_B120N10_Baby_Monitor',
 'SimpleHome_XCS7_1002_WHT_Security_Camera']

same autoencoder as the baseline notebook, keeping it identical so the comparison
at the end is actually fair

In [4]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

INPUT_DIM = len(feature_cols)
global_scaler = StandardScaler().fit(df[df["label"] == 0][feature_cols].values)

In [5]:
# splitting each device's data into its own local train/test set

def get_client_data(device_name):
    device_df = df[df["device"] == device_name]
    benign = device_df[device_df["label"] == 0][feature_cols].values
    attack = device_df[device_df["label"] == 1][feature_cols].values

    n_train = int(len(benign) * 0.7)
    train = global_scaler.transform(benign[:n_train])
    test_benign = benign[n_train:]

    X_test = np.vstack([test_benign, attack]) if len(attack) else test_benign
    y_test = np.hstack([np.zeros(len(test_benign)), np.ones(len(attack))])
    X_test = global_scaler.transform(X_test)

    return train, X_test, y_test

## flower client

each simulated client = one iot device. trains on its own local benign traffic,
sends weights back, never sends raw data anywhere

In [6]:
# standard flwr NumPyClient setup, fit() trains locally, evaluate() reports back auc

class IoTClient(fl.client.NumPyClient):
    def __init__(self, device_name):
        self.device_name = device_name
        self.model = Autoencoder(INPUT_DIM).to(device_torch)
        self.X_train, self.X_test, self.y_test = get_client_data(device_name)
        self.X_train_t = torch.tensor(self.X_train, dtype=torch.float32).to(device_torch)

    def get_parameters(self, config):
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        params = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        self.model.train()
        for _ in range(5):
            optimizer.zero_grad()
            recon = self.model(self.X_train_t)
            loss = criterion(recon, self.X_train_t)
            loss.backward()
            optimizer.step()
        return self.get_parameters({}), len(self.X_train), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        with torch.no_grad():
            X_test_t = torch.tensor(self.X_test, dtype=torch.float32).to(device_torch)
            recon = self.model(X_test_t)
            error = torch.mean((X_test_t - recon) ** 2, dim=1).cpu().numpy()
        auc = roc_auc_score(self.y_test, error) if len(set(self.y_test)) > 1 else 0.5
        return 0.0, len(self.X_test), {"roc_auc": auc, "device": self.device_name}

## running the simulation

10 rounds, all 4 devices every round

In [7]:
!pip install -q "flwr[simulation]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.0 MB/s eta 0:00:00


In [9]:
# this'll take a few min, training 4 local models x 10 rounds

def client_fn(cid):
    return IoTClient(DEVICES[int(cid)]).to_client()

def weighted_average(metrics):
    # metrics is a list of (num_examples, metrics_dict) tuples, one per client
    aucs = [m["roc_auc"] for _, m in metrics]
    devices = [m["device"] for _, m in metrics]
    per_device = dict(zip(devices, aucs))
    avg_auc = sum(aucs) / len(aucs)
    return {"avg_roc_auc": avg_auc, "per_device_auc": per_device}

strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=len(DEVICES),
    min_evaluate_clients=len(DEVICES),
    min_available_clients=len(DEVICES),
    evaluate_metrics_aggregation_fn=weighted_average,
)

history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=len(DEVICES),
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
)
history

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

          

History (loss, distributed):
	round 1: 0.0
	round 2: 0.0
	round 3: 0.0
	round 4: 0.0
	round 5: 0.0
	round 6: 0.0
	round 7: 0.0
	round 8: 0.0
	round 9: 0.0
	round 10: 0.0
History (metrics, distributed, evaluate):
{'avg_roc_auc': [(1, np.float64(0.9999731538305195)),
                 (2, np.float64(0.9999731562884945)),
                 (3, np.float64(0.9999733463193983)),
                 (4, np.float64(0.9999739367876712)),
                 (5, np.float64(0.9999750217548565)),
                 (6, np.float64(0.9999760375841669)),
                 (7, np.float64(0.9999758795945586)),
                 (8, np.float64(0.9999751172610711)),
                 (9, np.float64(0.9999742673172396)),
                 (10, np.float64(0.9999732525316699))],
 'per_device_auc': [(1,
                     {'Danmini_Doorbell': np.float64(0.9999972329025015),
                      'Ecobee_Thermostat': np.float64(0.9999993544469353),
                      'Philips_B120N10_Baby_Monitor': np.float64(0.999996

## federated vs centralized

pulling the centralized numbers back in to compare

In [10]:
import json

with open("/content/drive/MyDrive/nbaiot-project/processed/baseline_results.json") as f:
    baseline = json.load(f)

print("centralized isolation forest:", baseline["isolation_forest"]["roc_auc"])
print("centralized autoencoder:", baseline["autoencoder"]["roc_auc"])
print("federated autoencoder — see history.metrics_distributed above for per-round numbers")

centralized isolation forest: 0.9762494846705717
centralized autoencoder: 0.9999945608815071
federated autoencoder — see history.metrics_distributed above for per-round numbers


In [11]:
history.metrics_distributed

{'avg_roc_auc': [(1, np.float64(0.9999731538305195)),
  (2, np.float64(0.9999731562884945)),
  (3, np.float64(0.9999733463193983)),
  (4, np.float64(0.9999739367876712)),
  (5, np.float64(0.9999750217548565)),
  (6, np.float64(0.9999760375841669)),
  (7, np.float64(0.9999758795945586)),
  (8, np.float64(0.9999751172610711)),
  (9, np.float64(0.9999742673172396)),
  (10, np.float64(0.9999732525316699))],
 'per_device_auc': [(1,
   {'SimpleHome_XCS7_1002_WHT_Security_Camera': np.float64(0.9998991227467485),
    'Danmini_Doorbell': np.float64(0.9999972329025015),
    'Philips_B120N10_Baby_Monitor': np.float64(0.9999969052258927),
    'Ecobee_Thermostat': np.float64(0.9999993544469353)}),
  (2,
   {'Danmini_Doorbell': np.float64(0.9999972939430732),
    'Philips_B120N10_Baby_Monitor': np.float64(0.999996907863542),
    'Ecobee_Thermostat': np.float64(0.9999994021479007),
    'SimpleHome_XCS7_1002_WHT_Security_Camera': np.float64(0.999899021199462)}),
  (3,
   {'Philips_B120N10_Baby_Monitor

## Results

federated came out basically tied with centralized: 0.99998 avg roc-auc federated
vs 0.9999 centralized autoencoder. barely any gap, which is honestly the finding here
- going federated (private, no raw data leaving each device) didn't cost accuracy in
this setup

per device at round 10:
- danmini doorbell: 0.99999
- ecobee thermostat: 0.99999
- philips baby monitor: 0.99999
- simplehome security camera: 0.99989

camera's consistently a little behind the other three, every single round, though
still really strong overall. my guess is its "normal" traffic is just more varied
(different resolutions, motion triggered bursts etc) so it's a slightly harder
baseline to learn compared to something like a thermostat which probably has a much
more repetitive traffic pattern

also converged fast, avg auc barely moved after round 1, stayed roughly flat through
round 10. good sign for a real deployment since it means you wouldn't need many
communication rounds to get solid performance